# Half-page image cutter

For each page, process both half-page PNGs (L then R). Within each half,
run the row-by-row pixel procedure to find content blocks, ask
`claude-opus-4-7` per block whether it is an illustration or listing
text, then cut the original half at the top Y of every illustration block.

Slices are written directly into OUTPUT_DIR as
`page-NNNN-MMMM.png`, where NNNN is the page number and MMMM is a
1-based counter that runs across the L slices first, then the R slices,
preserving top-to-bottom reading order down the page. No per-half
subdirectories are created.


In [1]:
import os, io, base64
from pathlib import Path
import numpy as np
from PIL import Image
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()
client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])


In [2]:
# --- config ---
INPUT_DIR  = Path("./wip/cache/VA_ASCC_CTLG_halves")        # contains page-NNNN-L.png / page-NNNN-R.png
OUTPUT_DIR = Path("./wip/out/VA_ASCC_CTLG")        # cut slices land here, one subdir per input image
MODEL      = "claude-opus-4-7"

DARK_BRIGHTNESS_MAX  = 180   # pixel < this counts as dark
ROW_DARK_MIN_PIXELS  = 2     # row counts as non-blank if it has at least this many dark pixels
BLANK_RUN            = 5     # rows of blank to start/end a block
CENTER_FRACTION      = 0.90  # scan only the center 90% of width


In [3]:
def find_blocks(img_gray):
    """Step 1-2: scan rows, count dark pixels in center band, group into blocks.

    A block starts when a non-blank row appears after BLANK_RUN+ blank rows,
    and ends when BLANK_RUN+ blank rows follow.
    Returns a list of (y_top, y_bottom) inclusive tuples.
    """
    arr = np.array(img_gray)
    H, W = arr.shape

    margin = (1.0 - CENTER_FRACTION) / 2.0
    left   = int(W * margin)
    right  = int(W * (1.0 - margin))
    center = arr[:, left:right]

    dark_per_row = (center < DARK_BRIGHTNESS_MAX).sum(axis=1)
    is_dark = dark_per_row >= ROW_DARK_MIN_PIXELS

    blocks = []
    in_block = False
    block_start = None
    last_dark = None
    blank_run = BLANK_RUN  # primed so the first dark row opens a block

    for y in range(H):
        if is_dark[y]:
            if not in_block:
                if blank_run >= BLANK_RUN:
                    in_block = True
                    block_start = y
                    last_dark = y
            else:
                last_dark = y
            blank_run = 0
        else:
            blank_run += 1
            if in_block and blank_run >= BLANK_RUN:
                blocks.append((block_start, last_dark))
                in_block = False
                block_start = None

    if in_block:
        blocks.append((block_start, last_dark))

    return blocks


In [4]:
def classify_block(block_img):
    """Step 3: send a block image to Claude. Returns 'illustration' or 'text'."""
    buf = io.BytesIO()
    block_img.save(buf, format="PNG")
    b64 = base64.standard_b64encode(buf.getvalue()).decode("utf-8")

    resp = client.messages.create(
        model=MODEL,
        max_tokens=16,
        messages=[{
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "source": {"type": "base64", "media_type": "image/png", "data": b64},
                },
                {
                    "type": "text",
                    "text": (
                        "This is a horizontal strip from a scanned philatelic catalogue page "
                        "(one half-page wide). It contains exactly one of:\n"
                        "  illustration - a visual representation of a postmark, either a "
                        "manuscript mark or a typeset rendering of a handstamp\n"
                        "  text - listing rows, section banners, column headers, or running headers\n"
                        "Reply with exactly one word: illustration or text."
                    ),
                },
            ],
        }],
    )
    answer = resp.content[0].text.strip().lower()
    return "illustration" if "illustration" in answer else "text"


In [ ]:
def slice_page_half(img_path):
    """Run the find/classify/cut procedure on one half-page image.

    Returns (classifications, cut_ys, slices) where slices is a list of
    PIL.Image crops in top-to-bottom order. No files are written here;
    the caller is responsible for naming and saving across both halves.
    """
    img  = Image.open(img_path)
    gray = img.convert("L")
    W, H = img.size

    blocks = find_blocks(gray)

    classifications = []
    cut_ys = []
    for (y0, y1) in blocks:
        block_im = img.crop((0, y0, W, y1 + 1))
        kind = classify_block(block_im)
        classifications.append((y0, y1, kind))
        if kind == "illustration":
            cut_ys.append(y0)

    boundaries = [0] + cut_ys + [H]
    slices = []
    for i in range(len(boundaries) - 1):
        a, b = boundaries[i], boundaries[i + 1]
        if b <= a:
            continue
        slices.append(img.crop((0, a, W, b)))

    return classifications, cut_ys, slices


In [ ]:
# --- run ---
# Group input half-page PNGs by page number. For each page, process the
# L half then the R half, and write the resulting slices directly into
# OUTPUT_DIR with names like page-0419-0001.png. The counter runs
# across both halves so that the alphabetic order of the output files
# matches top-to-bottom reading order down the full page.
#
# Input filename convention (from the half-page splitter upstream):
#   page-NNNN-L.png
#   page-NNNN-R.png
# Example output for page 0419 with 12 L slices and 10 R slices:
#   page-0419-0001.png ... page-0419-0012.png  (from L half)
#   page-0419-0013.png ... page-0419-0022.png  (from R half)
import re

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
exts = {".png", ".PNG"}

pat = re.compile(r"^(page-\d+)-([LR])$")
pages = {}  # page_id (e.g. "page-0419") -> {"L": Path, "R": Path}
for p in sorted(INPUT_DIR.iterdir()):
    if p.suffix not in exts:
        continue
    m = pat.match(p.stem)
    if not m:
        print(f"  skip (no L/R suffix): {p.name}")
        continue
    page_id, side = m.group(1), m.group(2)
    pages.setdefault(page_id, {})[side] = p

for page_id in sorted(pages):
    halves = pages[page_id]
    print(f"--- {page_id} ---")
    counter = 1
    for side in ("L", "R"):
        if side not in halves:
            print(f"  missing {side} half")
            continue
        img_path = halves[side]
        classifications, cut_ys, slices = slice_page_half(img_path)
        n_blocks = len(classifications)
        n_illus  = len(cut_ys)
        n_slices = len(slices)
        print(f"  {side}: blocks={n_blocks} illustrations={n_illus} slices={n_slices}")
        for (y0, y1, kind) in classifications:
            print(f"    y=[{y0:5d}..{y1:5d}]  {kind}")
        for im in slices:
            out_path = OUTPUT_DIR / f"{page_id}-{counter:04d}.png"
            im.save(out_path)
            counter += 1

print("done.")
